# Gemini book embeddings setup

Prepare the database connection and inspect the book and embedding tables before implementing embedding ingestion. This notebook currently checks setup and schema; it does not generate or store embeddings.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from sqlalchemy import create_engine, text, inspect

SUPABASE_DATABASE_URL = os.getenv("SUPABASE_DATABASE_URL")
engine = create_engine(SUPABASE_DATABASE_URL)

## Check database connectivity

Run a lightweight query to confirm the database connection is available.

In [3]:
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("Connection successful:", result.scalar())

except Exception as e:
    print("Connection failed:", e)

Connection successful: 1


## Register the pgvector type

Teach SQLAlchemy to recognize PostgreSQL `vector` columns so their schema can be inspected correctly.

In [4]:
from pgvector.sqlalchemy import VECTOR
from sqlalchemy.dialects.postgresql.base import ischema_names

ischema_names["vector"] = VECTOR

## Inspect the destination tables

Review the columns, nullability, and foreign keys for `books` and `book_embeddings` before adding embedding records.

In [5]:
inspector = inspect(engine)

for table in ["books", "book_embeddings"]:
    print(f"\n{table.upper()}")

    for column in inspector.get_columns(table):
        print(
            column["name"],
            column["type"],
            "nullable:", column["nullable"]
        )

    print("Foreign keys:", inspector.get_foreign_keys(table))


BOOKS
id BIGINT nullable: False
isbn13 VARCHAR nullable: True
isbn10 VARCHAR nullable: True
title VARCHAR nullable: False
authors TEXT nullable: True
categories TEXT nullable: True
thumbnail TEXT nullable: True
description TEXT nullable: True
published_year INTEGER nullable: True
average_rating DOUBLE PRECISION nullable: True
num_pages INTEGER nullable: True
ratings_count INTEGER nullable: True
title_and_subtitle TEXT nullable: True
tagged_description TEXT nullable: True
simple_categories TEXT nullable: True
anger DOUBLE PRECISION nullable: True
disgust DOUBLE PRECISION nullable: True
fear DOUBLE PRECISION nullable: True
joy DOUBLE PRECISION nullable: True
sadness DOUBLE PRECISION nullable: True
surprise DOUBLE PRECISION nullable: True
neutral DOUBLE PRECISION nullable: True
Foreign keys: []

BOOK_EMBEDDINGS
id BIGINT nullable: False
book_id BIGINT nullable: False
model VARCHAR nullable: False
dimensions INTEGER nullable: False
embedding VECTOR(768) nullable: False
Foreign keys: [{'na

In [6]:
import pandas as pd

## Load book text for processing

Fetch each book ID, title, and description from the database as input for the embedding workflow.

In [7]:
with engine.connect() as conn:
    df_books = pd.read_sql(
        text("SELECT id, title, description FROM books"),
        conn
    )

df_books.sort_values("id").reset_index(drop=True)

,id,title,description
0,1,Gilead,A NOVEL THAT READERS and critics have been eag...
1,2,Spider's Web,A new 'Christie for Christmas' -- a full-lengt...
2,3,Rage of angels,"A memorable, mesmerizing heroine Jennifer -- b..."
3,4,The Four Loves,Lewis' work on the nature of love divides love...
4,5,The Problem of Pain,"""In The Problem of Pain, C.S. Lewis, one of th..."
...,...,...,...
5192,5193,Mistaken Identity,On A Train Journey Home To North India After L...
5193,5194,Journey to the East,This book tells the tale of a man who goes on ...
5194,5195,The Monk Who Sold His Ferrari: A Fable About F...,"Wisdom to Create a Life of Passion, Purpose, a..."
5195,5196,I Am that,This collection of the timeless teachings of o...


## Check for repeated descriptions

Find books that share an identical description so repeated text can be reviewed before embedding.

In [8]:
duplicates = df_books[
    df_books.duplicated(subset=["description"], keep=False)
].sort_values("description")

print(f"Books with duplicated descriptions: {len(duplicates)}")
print(f"Unique duplicated descriptions: {duplicates['description'].nunique()}")

duplicates

Books with duplicated descriptions: 72
Unique duplicated descriptions: 29


,id,title,description
280,275,The Alchemist - Gift Edition,"""My heart is afraid that it will have to suffe..."
370,367,The Alchemist,"""My heart is afraid that it will have to suffe..."
5025,5024,Death,"A tale of music, mortality, friendship and dea..."
4615,4620,Death,"A tale of music, mortality, friendship and dea..."
2579,2572,The Secret Garden,A ten-year-old orphan comes to live in a lonel...
...,...,...,...
4471,4467,The Story of the Life of Lafayette,This is a reproduction of the original artefac...
4098,4089,Arundhati Roy's The God of Small Things,This is part of a new series of guides to cont...
4095,4086,David Foster Wallace's Infinite Jest,This is part of a new series of guides to cont...
3560,3554,Spain,Three-dimensional cutaway illustrations and fl...


## Exclude boilerplate descriptions

Filter out descriptions containing the known reproduction notice to create the pending set for later processing.

In [9]:
boilerplate = "This is a reproduction of the original artefact."

df_pending = df_books[
    ~df_books["description"].str.contains(
        boilerplate,
        case=False,
        na=False,
        regex=False
    )
].copy()

df_pending.sort_values("id").reset_index(drop=True)

,id,title,description
0,1,Gilead,A NOVEL THAT READERS and critics have been eag...
1,2,Spider's Web,A new 'Christie for Christmas' -- a full-lengt...
2,3,Rage of angels,"A memorable, mesmerizing heroine Jennifer -- b..."
3,4,The Four Loves,Lewis' work on the nature of love divides love...
4,5,The Problem of Pain,"""In The Problem of Pain, C.S. Lewis, one of th..."
...,...,...,...
5186,5193,Mistaken Identity,On A Train Journey Home To North India After L...
5187,5194,Journey to the East,This book tells the tale of a man who goes on ...
5188,5195,The Monk Who Sold His Ferrari: A Fable About F...,"Wisdom to Create a Life of Passion, Purpose, a..."
5189,5196,I Am that,This collection of the timeless teachings of o...
